# Stock Analysis with Marketstack API in Python

In this notebook, we’ll:

1. Fetch historical **end-of-day (EOD)** stock data from the **Marketstack API**  
2. Load it into a **pandas DataFrame**  
3. Add several **technical indicators**:
   - Daily return (%)
   - Moving averages (MA20, MA50)
   - Rolling volatility
   - RSI (14-day)
   - MACD & Signal line
   - Cumulative returns
   - Drawdowns
   - Simple moving-average trading signal  
4. Save the final dataset to CSV


## 1. Imports and Basic Setup

Here we import:

- `os` – to read environment variables (for the API key)
- `requests` – to call the Marketstack REST API
- `pandas` – for data manipulation and analysis
- `datetime`/`timedelta` – to build a dynamic date range for our query
- `dotenv` – to load the API key from a `.env` file (optional but safer than hardcoding)


In [25]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv

# -------------------------------------------------
# Config
# -------------------------------------------------

# Load environment variables from .env if present
load_dotenv()

True

## 2. Configuration: API Key, Symbol, Date Range

We set up:

- `API_KEY` – your Marketstack API key  
  - Either stored in `.env` as `MARKETSTACK_API_KEY`
  - Or hardcoded (OK for learning, not for production)
- `DEFAULT_SYMBOL` – stock ticker to analyze (`AAPL` by default)
- `DAYS_BACK` – how many days of history to pull (e.g., 90 ≈ 3 months)
- `BASE_URL` – Marketstack base endpoint


In [26]:
# OPTION 1: Put your key in a .env file as MARKETSTACK_API_KEY=your_key_here
API_KEY = os.getenv("MARKETSTACK_API_KEY")

# OPTION 2: Hardcode your key here instead (less secure but simpler while learning)
# API_KEY = "YOUR_MARKETSTACK_API_KEY_HERE"

# Default settings (you can change these)
DEFAULT_SYMBOL = "AAPL"
DAYS_BACK = 90  # ~3 months of data
BASE_URL = "http://api.marketstack.com/v1"  # v1 EOD endpoint

## 3. Fetching EOD Data from Marketstack

### What this function does

`fetch_eod_data(symbol, days_back)`:

1. Builds a **date range**: from `today - days_back` up to `today`
2. Calls the `/eod` endpoint with:
   - `access_key` – your API key
   - `symbols` – ticker symbol (e.g., "AAPL")
   - `date_from`, `date_to` – the time window
3. Parses the JSON response and extracts the `"data"` list
4. Converts that list into a **pandas DataFrame**
5. Converts the `date` column to `datetime` and sorts by date

The output DataFrame typically includes columns like:
- `date`, `open`, `high`, `low`, `close`, `volume`, etc.


In [27]:

# -------------------------------------------------
# Core functions
# -------------------------------------------------

def fetch_eod_data(symbol: str, days_back: int = 90) -> pd.DataFrame:
    """
    Fetch end-of-day (EOD) data for a symbol over the last `days_back` days
    from the Marketstack API and return it as a pandas DataFrame.
    """

    if not API_KEY:
        raise ValueError("API key not found. Set MARKETSTACK_API_KEY env var or hardcode API_KEY.")

    # Calculate date range
    date_to = datetime.utcnow().date()
    date_from = date_to - timedelta(days=days_back)

    params = {
        "access_key": API_KEY,
        "symbols": symbol,
        "date_from": date_from.isoformat(),
        "date_to": date_to.isoformat(),
        "limit": 1000,  # just in case; free plan supports up to 12 months of history
    }

    url = f"{BASE_URL}/eod"
    response = requests.get(url, params=params)

    # Raise clear error if request fails (4xx / 5xx)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        print("Error response from Marketstack:")
        print(response.text)
        raise e

    data = response.json()

    if "data" not in data:
        # Marketstack sometimes returns an "error" object instead
        raise RuntimeError(f"Unexpected API response: {data}")

    records = data["data"]

    if not records:
        raise RuntimeError(f"No EOD data returned for symbol {symbol}")

    # Convert to DataFrame
    df = pd.DataFrame(records)

    # Convert date column and sort oldest -> newest
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    return df


def get_latest_close_price(df: pd.DataFrame) -> float:
    """
    From an EOD DataFrame, return the latest close price.
    Assumes df is sorted by date ascending.
    """
    latest_row = df.iloc[-1]
    return float(latest_row["close"])


def save_to_csv(df: pd.DataFrame, symbol: str, filename: str | None = None) -> str:
    """
    Save the DataFrame to CSV and return the file path.
    """
    if filename is None:
        filename = f"{symbol}_eod_data.csv"

    df.to_csv(filename, index=False)
    return filename


# -------------------------------------------------
# Simple analysis example
# -------------------------------------------------

def add_daily_returns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add a daily percentage return column based on the close price.
    """
    df = df.copy()
    df["daily_return_pct"] = df["close"].pct_change() * 100
    return df


# -------------------------------------------------
# Script entrypoint
# -------------------------------------------------

if __name__ == "__main__":
    # You can change this to any ticker you want
    symbol = DEFAULT_SYMBOL

    print(f"Fetching ~{DAYS_BACK} days of EOD data for {symbol} from Marketstack...")

    df_eod = fetch_eod_data(symbol, DAYS_BACK)
    df_eod = add_daily_returns(df_eod)

    latest_close = get_latest_close_price(df_eod)

    print(f"\n✅ Latest close price for {symbol}: {latest_close:.2f}")
    print("\nSample of the DataFrame:")
    print(df_eod.tail())

    csv_path = save_to_csv(df_eod, symbol)
    print(f"\n📁 Data saved to: {csv_path}")


Fetching ~90 days of EOD data for AAPL from Marketstack...


C:\Users\trevo\AppData\Local\Temp\ipykernel_29228\2329025258.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_to = datetime.utcnow().date()



✅ Latest close price for AAPL: 272.41

Sample of the DataFrame:
       open     high     low   close      volume  adj_high  adj_low  \
59  268.960  273.730  267.46  269.43  41282100.0   273.730  267.455   
60  269.810  275.845  269.80  275.25  45848160.0   275.910  269.800   
61  275.075  275.730  271.87  273.47  48058755.0   275.730  271.700   
62  274.110  276.700  272.09  272.95  49565800.0   276.699  272.090   
63  271.050  275.960  269.60  272.41  47399300.0   275.960  269.600   

    adj_close  adj_open  adj_volume  split_factor  dividend symbol exchange  \
59     269.43    268.96  41312412.0           1.0      0.26   AAPL     XNAS   
60     275.25    269.81  46208318.0           1.0      0.00   AAPL     XNAS   
61     273.47    275.00  48397982.0           1.0      0.00   AAPL     XNAS   
62     272.95    274.11  49602794.0           1.0      0.00   AAPL     XNAS   
63     272.41    271.05  47431331.0           1.0      0.00   AAPL     XNAS   

                        date  dai

## 4. Adding Technical Indicators

To keep your analysis organized and reusable, we create a function called `add_technical_indicators(df_eod)` that takes an end-of-day (EOD) price DataFrame and adds calculated technical indicators to it. This function returns a **new copy** of the DataFrame so the original data remains unchanged.

### What this function does
1. Accepts a DataFrame that includes at least a `close` column  
2. Calculates one or more technical indicators (in this example, a 10-day moving average)  
3. Adds these indicators as new columns  
4. Returns an enriched copy of the DataFrame  

### Why this approach is useful
- Keeps your code modular and clean  
- Makes it easy to apply the same analysis to different tickers or date ranges  
- Centralizes all indicator-calculation logic in one place for easier maintenance  


In [ ]:
import pandas as pd

def add_technical_indicators(df_eod: pd.DataFrame) -> pd.DataFrame:
    """
    Given a DataFrame with at least a 'close' column (and ideally 'high', 'low'),
    add common technical indicators and return an enriched copy.
    """
    df_eod = df_eod.copy()

    # Example indicator: 10-day moving average of close
    df_eod["MA_10"] = df_eod["close"].rolling(window=10).mean()

    # Make sure you RETURN the DataFrame
    return df_eod


if __name__ == "__main__":
    data = {
        "close": [1,2,3,4,5,6,7,8,9,10,11,12]
    }
    df = pd.DataFrame(data)

    df_with_indicators = add_technical_indicators(df)

    # Show something in the terminal
    print(df_with_indicators)


    close  MA_10
0       1    NaN
1       2    NaN
2       3    NaN
3       4    NaN
4       5    NaN
5       6    NaN
6       7    NaN
7       8    NaN
8       9    NaN
9      10    5.5
10     11    6.5
11     12    7.5


### 4.2 Moving Averages (MA20, MA50)

**Idea:**  
Moving averages smooth out price noise by averaging over the last N days.

- MA20 = average of last 20 closing prices
- MA50 = average of last 50 closing prices

These are used to detect trends:
- If MA20 > MA50 → recent prices are stronger than longer-term prices → bullish trend
- If MA20 < MA50 → bearish trend


In [29]:
df_eod["MA20"] = df_eod["close"].rolling(20).mean()
df_eod["MA50"] = df_eod["close"].rolling(50).mean()
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,split_factor,dividend,symbol,exchange,date,daily_return_pct,MA20,MA50
0,231.780,233.120,230.11,230.89,36051605.0,233.120,230.1100,230.89,231.700,37476188.0,1.0,0.00,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN
1,231.280,232.870,229.35,230.56,39320800.0,232.870,229.3500,230.56,231.275,39402564.0,1.0,0.00,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN
2,229.980,230.470,225.77,226.01,42190600.0,230.470,225.7700,226.01,229.980,42263865.0,1.0,0.00,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN
3,226.270,226.520,223.78,224.90,30593700.0,226.520,223.7804,224.90,226.270,30621249.0,1.0,0.00,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN
4,226.170,229.090,225.41,227.76,42445300.0,229.090,225.4100,227.76,226.170,42477811.0,1.0,0.00,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,268.960,273.730,267.46,269.43,41282100.0,273.730,267.4550,269.43,268.960,41312412.0,1.0,0.26,AAPL,XNAS,2025-11-10 00:00:00+00:00,0.357582,263.4445,252.9980
60,269.810,275.845,269.80,275.25,45848160.0,275.910,269.8000,275.25,269.810,46208318.0,1.0,0.00,AAPL,XNAS,2025-11-11 00:00:00+00:00,2.160116,264.8185,253.9086
61,275.075,275.730,271.87,273.47,48058755.0,275.730,271.7000,273.47,275.000,48397982.0,1.0,0.00,AAPL,XNAS,2025-11-12 00:00:00+00:00,-0.646685,266.0250,254.6086
62,274.110,276.700,272.09,272.95,49565800.0,276.699,272.0900,272.95,274.110,49602794.0,1.0,0.00,AAPL,XNAS,2025-11-13 00:00:00+00:00,-0.190149,267.3000,255.2720


### 4.3 Rolling Volatility (20-Day Standard Deviation)

**Idea:**  
Volatility measures how “bouncy” the returns are.

We take the **standard deviation** of `daily_return_pct` over a 20-day rolling window:

- High volatility → returns jump around a lot
- Low volatility → returns are more stable


In [30]:
df_eod["volatility_20"] = df_eod["daily_return_pct"].rolling(20).std()
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,split_factor,dividend,symbol,exchange,date,daily_return_pct,MA20,MA50,volatility_20
0,231.780,233.120,230.11,230.89,36051605.0,233.120,230.1100,230.89,231.700,37476188.0,1.0,0.00,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN,NaN
1,231.280,232.870,229.35,230.56,39320800.0,232.870,229.3500,230.56,231.275,39402564.0,1.0,0.00,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN,NaN
2,229.980,230.470,225.77,226.01,42190600.0,230.470,225.7700,226.01,229.980,42263865.0,1.0,0.00,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN,NaN
3,226.270,226.520,223.78,224.90,30593700.0,226.520,223.7804,224.90,226.270,30621249.0,1.0,0.00,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN,NaN
4,226.170,229.090,225.41,227.76,42445300.0,229.090,225.4100,227.76,226.170,42477811.0,1.0,0.00,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,268.960,273.730,267.46,269.43,41282100.0,273.730,267.4550,269.43,268.960,41312412.0,1.0,0.26,AAPL,XNAS,2025-11-10 00:00:00+00:00,0.357582,263.4445,252.9980,1.208143
60,269.810,275.845,269.80,275.25,45848160.0,275.910,269.8000,275.25,269.810,46208318.0,1.0,0.00,AAPL,XNAS,2025-11-11 00:00:00+00:00,2.160116,264.8185,253.9086,1.264030
61,275.075,275.730,271.87,273.47,48058755.0,275.730,271.7000,273.47,275.000,48397982.0,1.0,0.00,AAPL,XNAS,2025-11-12 00:00:00+00:00,-0.646685,266.0250,254.6086,1.290895
62,274.110,276.700,272.09,272.95,49565800.0,276.699,272.0900,272.95,274.110,49602794.0,1.0,0.00,AAPL,XNAS,2025-11-13 00:00:00+00:00,-0.190149,267.3000,255.2720,1.268497


### 4.4 RSI (Relative Strength Index, 14-Day)

**Idea:**  
RSI compares the magnitude of recent gains vs recent losses.

Steps:

1. Compute day-to-day price change: `delta = close_t - close_(t-1)`
2. Separate **gains** (positive deltas) and **losses** (negative deltas, made positive)
3. Compute average gain and average loss over a 14-day rolling window
4. Compute RS (Relative Strength) = avg_gain / avg_loss
5. Use the RSI formula:

\[
\text{RSI} = 100 - \frac{100}{1 + RS}
\]

Interpretation:
- RSI > 70 → overbought
- RSI < 30 → oversold


In [31]:
delta = df_eod["close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss
df_eod["RSI14"] = 100 - (100 / (1 + rs))
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,split_factor,dividend,symbol,exchange,date,daily_return_pct,MA20,MA50,volatility_20,RSI14
0,231.780,233.120,230.11,230.89,36051605.0,233.120,230.1100,230.89,231.700,37476188.0,1.0,0.00,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN
1,231.280,232.870,229.35,230.56,39320800.0,232.870,229.3500,230.56,231.275,39402564.0,1.0,0.00,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN,NaN,NaN
2,229.980,230.470,225.77,226.01,42190600.0,230.470,225.7700,226.01,229.980,42263865.0,1.0,0.00,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN,NaN,NaN
3,226.270,226.520,223.78,224.90,30593700.0,226.520,223.7804,224.90,226.270,30621249.0,1.0,0.00,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN,NaN,NaN
4,226.170,229.090,225.41,227.76,42445300.0,229.090,225.4100,227.76,226.170,42477811.0,1.0,0.00,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,268.960,273.730,267.46,269.43,41282100.0,273.730,267.4550,269.43,268.960,41312412.0,1.0,0.26,AAPL,XNAS,2025-11-10 00:00:00+00:00,0.357582,263.4445,252.9980,1.208143,64.267352
60,269.810,275.845,269.80,275.25,45848160.0,275.910,269.8000,275.25,269.810,46208318.0,1.0,0.00,AAPL,XNAS,2025-11-11 00:00:00+00:00,2.160116,264.8185,253.9086,1.264030,83.816425
61,275.075,275.730,271.87,273.47,48058755.0,275.730,271.7000,273.47,275.000,48397982.0,1.0,0.00,AAPL,XNAS,2025-11-12 00:00:00+00:00,-0.646685,266.0250,254.6086,1.290895,77.245979
62,274.110,276.700,272.09,272.95,49565800.0,276.699,272.0900,272.95,274.110,49602794.0,1.0,0.00,AAPL,XNAS,2025-11-13 00:00:00+00:00,-0.190149,267.3000,255.2720,1.268497,72.244181


### 4.5 MACD (12, 26, 9)

**Idea:**  
MACD uses **exponential moving averages (EMAs)** to measure momentum:

- Fast EMA (12-day)
- Slow EMA (26-day)
- MACD line = EMA12 − EMA26
- Signal line = EMA of MACD (9-day)

Interpretation:
- MACD crossing above Signal → bullish momentum
- MACD crossing below Signal → bearish momentum


In [32]:
exp1 = df_eod["close"].ewm(span=12, adjust=False).mean()
exp2 = df_eod["close"].ewm(span=26, adjust=False).mean()
df_eod["MACD"] = exp1 - exp2
df_eod["Signal"] = df_eod["MACD"].ewm(span=9, adjust=False).mean()
df_eod

,open,high,low,close,volume,adj_high,adj_low,adj_close,adj_open,adj_volume,...,symbol,exchange,date,daily_return_pct,MA20,MA50,volatility_20,RSI14,MACD,Signal
0,231.780,233.120,230.11,230.89,36051605.0,233.120,230.1100,230.89,231.700,37476188.0,...,AAPL,XNAS,2025-08-18 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
1,231.280,232.870,229.35,230.56,39320800.0,232.870,229.3500,230.56,231.275,39402564.0,...,AAPL,XNAS,2025-08-19 00:00:00+00:00,-0.142925,NaN,NaN,NaN,NaN,-0.026325,-0.005265
2,229.980,230.470,225.77,226.01,42190600.0,230.470,225.7700,226.01,229.980,42263865.0,...,AAPL,XNAS,2025-08-20 00:00:00+00:00,-1.973456,NaN,NaN,NaN,NaN,-0.409613,-0.086134
3,226.270,226.520,223.78,224.90,30593700.0,226.520,223.7804,224.90,226.270,30621249.0,...,AAPL,XNAS,2025-08-21 00:00:00+00:00,-0.491129,NaN,NaN,NaN,NaN,-0.793788,-0.227665
4,226.170,229.090,225.41,227.76,42445300.0,229.090,225.4100,227.76,226.170,42477811.0,...,AAPL,XNAS,2025-08-22 00:00:00+00:00,1.271676,NaN,NaN,NaN,NaN,-0.857587,-0.353650
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,268.960,273.730,267.46,269.43,41282100.0,273.730,267.4550,269.43,268.960,41312412.0,...,AAPL,XNAS,2025-11-10 00:00:00+00:00,0.357582,263.4445,252.9980,1.208143,64.267352,5.193240,5.432199
60,269.810,275.845,269.80,275.25,45848160.0,275.910,269.8000,275.25,269.810,46208318.0,...,AAPL,XNAS,2025-11-11 00:00:00+00:00,2.160116,264.8185,253.9086,1.264030,83.816425,5.425069,5.430773
61,275.075,275.730,271.87,273.47,48058755.0,275.730,271.7000,273.47,275.000,48397982.0,...,AAPL,XNAS,2025-11-12 00:00:00+00:00,-0.646685,266.0250,254.6086,1.290895,77.245979,5.402882,5.425195
62,274.110,276.700,272.09,272.95,49565800.0,276.699,272.0900,272.95,274.110,49602794.0,...,AAPL,XNAS,2025-11-13 00:00:00+00:00,-0.190149,267.3000,255.2720,1.268497,72.244181,5.282446,5.396645


### 4.6 Cumulative Returns

**Idea:**  
If you invested $1 at the start and held the stock, how much would it be worth over time?

We compound daily returns:

\[
\text{cum\_return}_t = \prod_{i=1}^{t} (1 + \text{daily\_return}_i)
\]

Where `daily_return` is in decimal form (e.g., 2% → 0.02).


In [35]:
    # 6. Cumulative returns (growth of $1 invested at the start)
df_eod["cumulative_return"] = (1 + df_eod["daily_return_pct"] / 100).cumprod()
print(df_eod)

       open     high     low   close      volume  adj_high   adj_low  \
0   231.780  233.120  230.11  230.89  36051605.0   233.120  230.1100   
1   231.280  232.870  229.35  230.56  39320800.0   232.870  229.3500   
2   229.980  230.470  225.77  226.01  42190600.0   230.470  225.7700   
3   226.270  226.520  223.78  224.90  30593700.0   226.520  223.7804   
4   226.170  229.090  225.41  227.76  42445300.0   229.090  225.4100   
..      ...      ...     ...     ...         ...       ...       ...   
59  268.960  273.730  267.46  269.43  41282100.0   273.730  267.4550   
60  269.810  275.845  269.80  275.25  45848160.0   275.910  269.8000   
61  275.075  275.730  271.87  273.47  48058755.0   275.730  271.7000   
62  274.110  276.700  272.09  272.95  49565800.0   276.699  272.0900   
63  271.050  275.960  269.60  272.41  47399300.0   275.960  269.6000   

    adj_close  adj_open  adj_volume  ...  exchange                      date  \
0      230.89   231.700  37476188.0  ...      XNAS 2025

### 4.7 Drawdown (Drop From Historical Peak)

**Idea:**  
Drawdown measures how far the current price has fallen from the highest price seen so far.

Steps:

1. Compute running peak of `close`: `peak_t = max(close_1, ..., close_t)`
2. Drawdown = (close_t − peak_t) / peak_t × 100

- At peaks, drawdown = 0
- After drops, drawdown is negative (e.g., −10%, −25%)


In [39]:
    # 7. Drawdown (percentage drop from the historical peak at each point)df_eod["peak"] = df_eod["close"].cummax()
    df_eod["drawdown"] = (df_eod["close"] - df_eod["peak"]) / df["peak"] * 100
    print(df_eod)

       open     high     low   close      volume  adj_high   adj_low  \
0   231.780  233.120  230.11  230.89  36051605.0   233.120  230.1100   
1   231.280  232.870  229.35  230.56  39320800.0   232.870  229.3500   
2   229.980  230.470  225.77  226.01  42190600.0   230.470  225.7700   
3   226.270  226.520  223.78  224.90  30593700.0   226.520  223.7804   
4   226.170  229.090  225.41  227.76  42445300.0   229.090  225.4100   
..      ...      ...     ...     ...         ...       ...       ...   
59  268.960  273.730  267.46  269.43  41282100.0   273.730  267.4550   
60  269.810  275.845  269.80  275.25  45848160.0   275.910  269.8000   
61  275.075  275.730  271.87  273.47  48058755.0   275.730  271.7000   
62  274.110  276.700  272.09  272.95  49565800.0   276.699  272.0900   
63  271.050  275.960  269.60  272.41  47399300.0   275.960  269.6000   

    adj_close  adj_open  adj_volume  ...  daily_return_pct      MA20  \
0      230.89   231.700  37476188.0  ...               NaN     

### 4.8 Simple Moving-Average Trading Signal

**Idea:**  
We create a very simple trend-following signal:

- `signal = 1` → bullish (MA20 > MA50)
- `signal = -1` → bearish (MA20 < MA50)
- `signal = 0` → neutral

You could later backtest this logic (e.g., only “holding” the stock when signal = 1).


In [42]:
def add_technical_indicators(df_eod):

    # 8. Simple moving-average based trading signal
    df_eod["MA20"] = df_eod["close"].rolling(20).mean()
    df_eod["MA50"] = df_eod["close"].rolling(50).mean()

    df_eod["signal"] = 0
    df_eod.loc[df_eod["MA20"] > df_eod["MA50"], "signal"] = 1
    df_eod.loc[df_eod["MA20"] < df_eod["MA50"], "signal"] = -1

    return df_eod
print(df_eod)

       open     high     low   close      volume  adj_high   adj_low  \
0   231.780  233.120  230.11  230.89  36051605.0   233.120  230.1100   
1   231.280  232.870  229.35  230.56  39320800.0   232.870  229.3500   
2   229.980  230.470  225.77  226.01  42190600.0   230.470  225.7700   
3   226.270  226.520  223.78  224.90  30593700.0   226.520  223.7804   
4   226.170  229.090  225.41  227.76  42445300.0   229.090  225.4100   
..      ...      ...     ...     ...         ...       ...       ...   
59  268.960  273.730  267.46  269.43  41282100.0   273.730  267.4550   
60  269.810  275.845  269.80  275.25  45848160.0   275.910  269.8000   
61  275.075  275.730  271.87  273.47  48058755.0   275.730  271.7000   
62  274.110  276.700  272.09  272.95  49565800.0   276.699  272.0900   
63  271.050  275.960  269.60  272.41  47399300.0   275.960  269.6000   

    adj_close  adj_open  adj_volume  ...      MA20      MA50 volatility_20  \
0      230.89   231.700  37476188.0  ...       NaN       

## 5. Saving Data to CSV

Once all indicators are added, we may want to save the DataFrame to a CSV file
for use in Excel, Power BI, or other tools.


In [43]:
def save_to_csv(df: pd.DataFrame, symbol: str, filename: str | None = None) -> str:
    """
    Save the DataFrame to a CSV file and return the file path.
    """
    if filename is None:
        filename = f"{symbol}_eod_analysis.csv"

    df.to_csv(filename, index=False)
    return filename


## 6. Generating a Summary Report

We can also compute some summary statistics to get a feel for the stock:

- Average close
- Highest and lowest close
- Average daily return
- Average rolling volatility
- Maximum drawdown (worst drop from peak)


In [ ]:
def get_summary_stats(df_eod: pd.DataFrame) -> dict:
    """
    Return a dictionary of useful summary statistics for the analyzed DataFrame.
    Assumes indicators such as 'daily_return_pct' and 'drawdown' already exist.
    """
    summary = {
        "avg_close": df_eod["close"].mean(),
        "highest_close": df_eod["close"].max(),
        "lowest_close": df_eod["close"].min(),
        "avg_daily_return_pct": df_eod["daily_return_pct"].mean(),
        "avg_volatility_20": df_eod["volatility_20"].mean(),
        "max_drawdown_pct": df_eod["drawdown"].min(),  # most negative drawdown
    }
    return summary


## 7. Putting It All Together in a Main Script

This block:

1. Sets the symbol  
2. Fetches EOD data  
3. Adds all technical indicators  
4. Prints a tail of the DataFrame  
5. Prints summary stats  
6. Saves the full dataset to CSV


In [49]:
import pandas as pd

def add_technical_indicators(df_eod: pd.DataFrame) -> pd.DataFrame:
    """
    Given a DataFrame with at least a 'close' column (and ideally 'high', 'low'),
    add common technical indicators and return an enriched copy.
    """
    df = df_eod.copy()

    # 1. Daily % return (this is what was missing!)
    df["daily_return_pct"] = df["close"].pct_change() * 100

    # 2. Cumulative returns (growth of $1 invested at the start)
    df["cumulative_return"] = (1 + df["daily_return_pct"] / 100).cumprod()

    # 3. Moving averages
    df["MA20"] = df["close"].rolling(20).mean()
    df["MA50"] = df["close"].rolling(50).mean()

    # 4. Simple moving-average based trading signal
    df["signal"] = 0
    df.loc[df["MA20"] > df["MA50"], "signal"] = 1
    df.loc[df["MA20"] < df["MA50"], "signal"] = -1

    return df


def get_summary_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return summary statistics for key numeric columns.
    Only uses columns that actually exist in the DataFrame.
    """
    target_cols = ["close", "daily_return_pct", "cumulative_return"]
    existing_cols = [c for c in target_cols if c in df.columns]
    return df[existing_cols].describe()


if __name__ == "__main__":
    symbol = DEFAULT_SYMBOL  # change to any ticker you like, e.g., "MSFT", "TSLA"

    print(f"Fetching ~{DAYS_BACK} days of EOD data for {symbol} from Marketstack...")

    # 1. Fetch raw EOD data
    df_eod = fetch_eod_data(symbol, DAYS_BACK)

    # 2. Add indicators (this now creates daily_return_pct, cumulative_return, etc.)
    df_analysis = add_technical_indicators(df_eod)

    # 3. Show last few rows
    print("\nSample of the analyzed DataFrame:")
    print(df_analysis.tail())

    # 4. Summary statistics
    summary = get_summary_stats(df_analysis)
    print("\nSummary statistics:")
    print(summary)

    # 5. Save to CSV
    csv_path = save_to_csv(df_analysis, symbol)
    print(f"\nData with indicators saved to: {csv_path}")


Fetching ~90 days of EOD data for AAPL from Marketstack...


C:\Users\trevo\AppData\Local\Temp\ipykernel_29228\2329025258.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  date_to = datetime.utcnow().date()



Sample of the analyzed DataFrame:
       open     high     low   close      volume  adj_high  adj_low  \
59  268.960  273.730  267.46  269.43  41282100.0   273.730  267.455   
60  269.810  275.845  269.80  275.25  45848160.0   275.910  269.800   
61  275.075  275.730  271.87  273.47  48058755.0   275.730  271.700   
62  274.110  276.700  272.09  272.95  49565800.0   276.699  272.090   
63  271.050  275.960  269.60  272.41  47399300.0   275.960  269.600   

    adj_close  adj_open  adj_volume  split_factor  dividend symbol exchange  \
59     269.43    268.96  41312412.0           1.0      0.26   AAPL     XNAS   
60     275.25    269.81  46208318.0           1.0      0.00   AAPL     XNAS   
61     273.47    275.00  48397982.0           1.0      0.00   AAPL     XNAS   
62     272.95    274.11  49602794.0           1.0      0.00   AAPL     XNAS   
63     272.41    271.05  47431331.0           1.0      0.00   AAPL     XNAS   

                        date  daily_return_pct  cumulative_retu